<a href="https://colab.research.google.com/github/kirti73/AI_lab/blob/main/1BF24CS140_WEEK7_UNIFICATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
class Variable:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return self.name

    def __repr__(self):
        return f"Var({self.name})"

    def __eq__(self, other):
        return isinstance(other, Variable) and self.name == other.name

    def __hash__(self):
        return hash(self.name)


class Constant:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return self.name

    def __repr__(self):
        return f"Const({self.name})"

    def __eq__(self, other):
        return isinstance(other, Constant) and self.name == other.name

    def __hash__(self):
        return hash(self.name)


class Compound:
    def __init__(self, op, args):
        self.op = op  # Predicate or Function symbol
        self.args = list(args)

    def __str__(self):
        return f"{self.op}({', '.join(str(arg) for arg in self.args)})"

    def __repr__(self):
        return f"Compound({self.op}, {self.args})"

    def __eq__(self, other):
        return isinstance(other, Compound) and self.op == other.op and self.args == other.args

In [6]:
def occurs_check(var, term, subst):
    """Return True if variable var occurs anywhere inside term (under subst)."""
    if var == term:
        return True
    elif isinstance(term, Variable) and term in subst:
        return occurs_check(var, subst[term], subst)
    elif isinstance(term, Compound):
        return any(occurs_check(var, arg, subst) for arg in term.args)
    return False


def unify_var(var, val, subst):
    """Unifies a variable with a value while updating the substitution dict."""
    if var in subst:
        return unify(subst[var], val, subst)
    elif isinstance(val, Variable) and val in subst:
        return unify(var, subst[val], subst)
    elif occurs_check(var, val, subst):
        return None  # Unification fails due to occurs-check violation
    else:
        new_subst = subst.copy()
        new_subst[var] = val
        return new_subst


def unify(x, y, subst=None):
    """Finds a substitution that unifies terms x and y. Returns None on failure."""
    if subst is None:
        subst = {}

    if subst is None:
        return None

    # 1. If terms are identical
    if x == y:
        return subst

    # 2. If x is a variable
    elif isinstance(x, Variable):
        return unify_var(x, y, subst)

    # 3. If y is a variable
    elif isinstance(y, Variable):
        return unify_var(y, x, subst)

    # 4. If both are compound terms (Predicates or Functions)
    elif isinstance(x, Compound) and isinstance(y, Compound):
        if x.op != y.op or len(x.args) != len(y.args):
            return None  # Arity or symbol mismatch

        # Unify arguments recursively
        current_subst = subst
        for arg_x, arg_y in zip(x.args, y.args):
            current_subst = unify(arg_x, arg_y, current_subst)
            if current_subst is None:
                return None
        return current_subst

    # 5. Constant mismatch
    else:
        return None

In [15]:
# Variables
x = Variable('x')
y = Variable('y')
z = Variable('z')

# Constants
a = Constant('a')

b = Constant('b')

def resolve_term(term, subst):
    """Recursively apply substitutions to a term until no more substitutions can be made."""
    if isinstance(term, Variable):
        if term in subst:
            return resolve_term(subst[term], subst)
        return term
    elif isinstance(term, Compound):
        return Compound(term.op, [resolve_term(arg, subst) for arg in term.args])
    return term

# Helper function to print results clearly
def print_unify_result(expr1, expr2):
    subst = unify(expr1, expr2)
    if subst is not None:
        # Fully resolve all values in the substitution dictionary
        fully_resolved = {}
        for k in subst.keys():
            fully_resolved[str(k)] = str(resolve_term(k, subst))
        print(f"SUCCESS: Unify( {expr1}, {expr2} )\nFinal replacements for all variables: {fully_resolved}")
    else:
        print(f"FAILED: Unify( {expr1}, {expr2} ) fails")

# Case 1: Unify P(x, h(y)) and P(a, f(z)) -> Should fail (different function symbols 'h' and 'f')
h_y = Compound('h', [y])
f_z = Compound('f', [z])
expr1 = Compound('P', [x, h_y])
expr2 = Compound('P', [a, f_z])
print_unify_result(expr1, expr2)

# Case 2: Unify P(x, y) and P(a, b) -> Should succeed: {x: a, y: b}
expr3 = Compound('P', [x, y])
expr4 = Compound('P', [a, b])
print_unify_result(expr3, expr4)

# Case 3: Occurs check failure Unify(x, f(x))
f_x = Compound('f', [x])
print_unify_result(x, f_x)

FAILED: Unify( P(x, h(y)), P(a, f(z)) ) fails
SUCCESS: Unify( P(x, y), P(a, b) )
Final replacements for all variables: {'x': 'a', 'y': 'b'}
FAILED: Unify( x, f(x) ) fails


In [8]:
# Problem 1: Unify {P(f(a), g(y)), P(X, X)}
# Note: 'X' is capitalized, representing a Variable. 'a' is a Constant.
# We represent this as unifying x1 = P(f(a), g(y)) and x2 = P(X, X)
X = Variable('X')
y = Variable('y')
a = Constant('a')

f_a = Compound('f', [a])
g_y = Compound('g', [y])

expr1_a = Compound('P', [f_a, g_y])
expr1_b = Compound('P', [X, X])

print("--- Problem 1 ---")
print_unify_result(expr1_a, expr1_b)

# Problem 2: Unify {Q(a, g(x, a), f(y)), Q(a, g(f(b), a), x)}
x = Variable('x')
y = Variable('y')
a = Constant('a')
b = Constant('b')

g_x_a = Compound('g', [x, a])
f_y = Compound('f', [y])

f_b = Compound('f', [b])
g_fb_a = Compound('g', [f_b, a])

expr2_a = Compound('Q', [a, g_x_a, f_y])
expr2_b = Compound('Q', [a, g_fb_a, x])

print("\n--- Problem 2 ---")
print_unify_result(expr2_a, expr2_b)

# Problem 3: Unify {P(b, x, f(g(z))), P(z, f(y), f(y))}
x = Variable('x')
y = Variable('y')
z = Variable('z')
b = Constant('b')

g_z = Compound('g', [z])
fg_z = Compound('f', [g_z])

f_y = Compound('f', [y])

expr3_a = Compound('P', [b, x, fg_z])
expr3_b = Compound('P', [z, f_y, f_y])

print("\n--- Problem 3 ---")
print_unify_result(expr3_a, expr3_b)

--- Problem 1 ---
FAILED: Unify( P(f(a), g(y)), P(X, X) ) fails

--- Problem 2 ---
SUCCESS: Unify( Q(a, g(x, a), f(y)), Q(a, g(f(b), a), x) ) -> {'x': 'f(b)', 'y': 'b'}

--- Problem 3 ---
SUCCESS: Unify( P(b, x, f(g(z))), P(z, f(y), f(y)) ) -> {'z': 'b', 'x': 'f(y)', 'y': 'g(z)'}


In [ ]:
def parse_term(term_str):
    term_str = term_str.strip()
    if '(' not in term_str:
        # Capitals / Capital-starting words are Variables, lowercase are Constants
        if term_str[0].isupper():
            return Variable(term_str)
        else:
            return Constant(term_str)

    op_end = term_str.index('(')
    op = term_str[:op_end].strip()

    # Slice out the arguments inside the outermost matching parentheses
    args_str = term_str[op_end+1:-1].strip()
    args = []
    bracket_count = 0
    current_arg = []

    for char in args_str:
        if char == ',' and bracket_count == 0:
            args.append(parse_term(''.join(current_arg)))
            current_arg = []
        else:
            if char == '(':
                bracket_count += 1
            elif char == ')':
                bracket_count -= 1
            current_arg.append(char)

    if current_arg:
        args.append(parse_term(''.join(current_arg)))

    return Compound(op, args)

# Take user input interactively
try:
    input_1 = input("Enter the first term: ")
    input_2 = input("Enter the second term: ")

    term_1 = parse_term(input_1)
    term_2 = parse_term(input_2)

    print_unify_result(term_1, term_2)
except Exception as e:
    print(f"Error parsing inputs: {e}. Please ensure inputs are structured correctly.")